# LangChain y LangSmith · RAG avanzado

Continuación de *RAG desde cero* con las partes 10 a 18 de [RAG From Scratch](https://github.com/langchain-ai/rag-from-scratch): **enrutamiento** (lógico y semántico), **construcción de consultas** con filtros de metadatos, **indexación** avanzada (multirrepresentación, RAPTOR y ColBERT), **reordenación** y flujos que se corrigen a sí mismos (CRAG y Self-RAG) construidos como grafos de [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview).

La celda de preparación repite el corpus, los *embeddings* locales y el modelo local del notebook anterior para que este se pueda ejecutar por separado. Igual que allí, lo que haría un LLM real (clasificar, extraer filtros, resumir, juzgar relevancia, reescribir) se simula con reglas deterministas.

In [1]:
import json
import re
import unicodedata
from collections import Counter

import numpy as np
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.feature_extraction.text import HashingVectorizer

ARTICULOS = [
    ("Agentes basados en LLM", "agentes", 2023, "Ana Pérez", 1200,
     "Un agente autónomo usa un modelo de lenguaje como cerebro. El agente combina planificación, memoria y uso de herramientas para resolver tareas largas. El modelo decide en cada paso qué acción ejecutar y observa el resultado antes de continuar."),
    ("Planificación y descomposición de tareas", "agentes", 2023, "Ana Pérez", 950,
     "La planificación divide una tarea compleja en subtareas más pequeñas y manejables. La cadena de pensamiento pide al modelo que razone paso a paso antes de responder. El árbol de pensamientos explora varias alternativas de razonamiento en cada paso y evalúa cuál conviene seguir."),
    ("Reflexión y autocrítica", "agentes", 2024, "Luis Santana", 610,
     "La reflexión permite a un agente corregir sus errores a partir de la experiencia. ReAct intercala pensamientos, acciones y observaciones en un mismo bucle. Reflexion guarda en memoria un resumen de los fallos anteriores para no repetirlos en el siguiente intento."),
    ("Memoria de los agentes", "agentes", 2024, "Luis Santana", 780,
     "La memoria a corto plazo es el contexto de la conversación que cabe en la ventana del modelo. La memoria a largo plazo guarda información en un almacén vectorial externo y la recupera por similitud. Los algoritmos de vecinos más cercanos aproximados, como HNSW o FAISS, aceleran esa búsqueda en millones de vectores."),
    ("Uso de herramientas", "agentes", 2025, "Carmen Díaz", 1500,
     "Las herramientas amplían lo que un modelo puede hacer: calculadoras, buscadores, intérpretes de código o APIs. El modelo genera una llamada a herramienta con argumentos estructurados y el sistema la ejecuta. Toolformer y MRKL son dos propuestas tempranas que enseñaron a los modelos a usar herramientas."),
    ("Qué es RAG", "rag", 2024, "Carmen Díaz", 2100,
     "La generación aumentada por recuperación, o RAG, conecta un modelo de lenguaje con documentos externos. Primero se recuperan los fragmentos relevantes para la pregunta y después el modelo genera la respuesta usando ese contexto. RAG reduce las alucinaciones y permite usar información privada o reciente sin reentrenar el modelo."),
    ("Indexación y troceado", "rag", 2024, "Pedro Rodríguez", 870,
     "Para indexar, los documentos se dividen en fragmentos de unos cientos de caracteres con cierto solapamiento. Cada fragmento se convierte en un vector numérico mediante un modelo de embeddings. Los vectores se guardan en un almacén vectorial junto con el texto y sus metadatos."),
    ("Recuperación por similitud", "rag", 2025, "Pedro Rodríguez", 640,
     "La recuperación compara el vector de la pregunta con los vectores de los fragmentos. La similitud del coseno mide el ángulo entre dos vectores y es la métrica más habitual. La búsqueda por máxima relevancia marginal, MMR, equilibra la relevancia con la diversidad de los resultados."),
    ("Transformación de consultas", "rag", 2025, "Ana Pérez", 520,
     "Una pregunta mal formulada recupera documentos poco útiles. Multi-query genera varias versiones de la pregunta y une los resultados. RAG-Fusion ordena los resultados combinados con la fusión de rangos recíprocos. HyDE genera un documento hipotético que responde a la pregunta y busca con su vector."),
    ("Reordenación y RAG correctivo", "rag", 2025, "Luis Santana", 410,
     "Un reordenador vuelve a puntuar los fragmentos recuperados con un modelo más preciso y se queda con los mejores. El RAG correctivo, CRAG, evalúa si los documentos recuperados son relevantes. Si no lo son, reescribe la consulta y busca en la web antes de generar la respuesta."),
    ("Evaluación de aplicaciones LLM", "evaluacion", 2025, "Carmen Díaz", 990,
     "Evaluar una aplicación con modelos de lenguaje exige un conjunto de ejemplos con entradas y salidas de referencia. Los evaluadores pueden ser reglas sencillas, métricas como la exactitud o un modelo que actúa como juez. LangSmith guarda los experimentos para comparar versiones de la aplicación."),
    ("Trazas y observabilidad", "evaluacion", 2026, "Pedro Rodríguez", 330,
     "Una traza registra cada paso de una aplicación: las llamadas al modelo, las herramientas y los recuperadores. Cada paso guarda sus entradas, salidas, duración y errores. Con las trazas se depuran fallos y se detectan los pasos más lentos o costosos."),
]

VACIAS = set("""a al algo ante antes como con cual cuales de del desde donde el ella ellos en entre es esa ese esta este esto
hay la las le les lo los mas me mi muy no o otra otro para pero poco por que se sin sobre son su sus tambien un una uno unos
unas y ya ha han hace cada cierto ese eso esos sea ser si""".split())


def sin_tildes(texto):
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")


def tokens(texto):
    palabras = re.findall(r"\w+", sin_tildes(texto.lower()))
    return [p[:5] for p in palabras if p not in VACIAS and len(p) > 1]


class EmbeddingsLocales(Embeddings):
    def __init__(self, dimension=2**12):
        self.vectorizador = HashingVectorizer(analyzer=tokens, n_features=dimension, alternate_sign=False, norm="l2")

    def embed_documents(self, textos):
        return self.vectorizador.transform(textos).toarray().tolist()

    def embed_query(self, texto):
        return self.embed_documents([texto])[0]


def frases_de(texto):
    return [f.strip() for f in re.split(r"(?<=[.!?])\s+", texto) if f.strip()]


def responder_con_contexto(prompt):
    partes = re.search(r"Contexto:\s*(.*?)\s*Pregunta:\s*(.*)", prompt, re.DOTALL)
    if not partes:
        return "No lo sé."
    clave = set(tokens(partes.group(2)))
    puntuadas = [(len(clave & set(tokens(f))), -i, f) for i, f in enumerate(frases_de(partes.group(1)))]
    mejor = max(puntuadas, default=(0, 0, ""))
    return mejor[2] if mejor[0] else "No lo sé."


class ModeloLocal(BaseChatModel):
    reglas: list = []

    @property
    def _llm_type(self) -> str:
        return "local"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        texto = messages[-1].text
        respuesta = next(
            (funcion(m) for patron, funcion in self.reglas if (m := re.search(patron, texto, re.IGNORECASE | re.DOTALL))),
            None,
        )
        respuesta = respuesta if respuesta is not None else responder_con_contexto(texto)
        return ChatResult(generations=[ChatGeneration(message=AIMessage(respuesta))])


def formatear(docs):
    return "\n\n".join(d.page_content for d in docs)


embeddings = EmbeddingsLocales()
documentos = [
    Document(page_content=texto, metadata={"id": f"d{i:02d}", "titulo": titulo, "tema": tema, "anio": anio, "autor": autor, "vistas": vistas})
    for i, (titulo, tema, anio, autor, vistas, texto) in enumerate(ARTICULOS, 1)
]
trozos = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=40, separators=["\n\n", ". ", " ", ""], keep_separator="end"
).split_documents(documentos)
almacen = InMemoryVectorStore.from_documents(trozos, embeddings)
recuperador = almacen.as_retriever(search_kwargs={"k": 4})
prompt_rag = ChatPromptTemplate.from_template("Responde usando solo el contexto.\n\nContexto:\n{contexto}\n\nPregunta: {pregunta}")
print(len(documentos), "documentos,", len(trozos), "fragmentos")

12 documentos, 25 fragmentos


## 1. Enrutamiento lógico

Crear un almacén vectorial por tema (`agentes`, `rag`, `evaluacion`) y un enrutador que pida al modelo la fuente adecuada en formato JSON, validada con un modelo de Pydantic `Ruta` cuyo campo es un `Literal`. El modelo local decide contando palabras clave de cada tema. Encadenar el enrutador con la recuperación en el almacén elegido y probarlo con tres preguntas.

In [2]:
from typing import Literal

from pydantic import BaseModel, Field


class Ruta(BaseModel):
    fuente: Literal["agentes", "rag", "evaluacion"] = Field(description="Colección de documentos más adecuada para la pregunta")


CLAVES_TEMA = {
    "agentes": ["agente", "planific", "memoria", "herramienta", "reflexi", "react", "razon"],
    "rag": ["rag", "recuper", "indexa", "fragmento", "embedding", "consulta", "reorden", "vector"],
    "evaluacion": ["evalu", "traza", "experimento", "juez", "langsmith", "metrica", "depura"],
}


def elegir_fuente(m):
    pregunta = sin_tildes(m.group(1).lower())
    puntos = {tema: sum(c in pregunta for c in claves) for tema, claves in CLAVES_TEMA.items()}
    return json.dumps({"fuente": max(puntos, key=puntos.get)})


parser_ruta = PydanticOutputParser(pydantic_object=Ruta)
prompt_ruta = ChatPromptTemplate.from_template("Elige la fuente de datos más adecuada.\n{formato}\n\nPregunta: {pregunta}").partial(
    formato=parser_ruta.get_format_instructions()
)
enrutador = prompt_ruta | ModeloLocal(reglas=[(r"Elige la fuente.*Pregunta:\s*(.*)", elegir_fuente)]) | parser_ruta

almacenes = {
    tema: InMemoryVectorStore.from_documents([t for t in trozos if t.metadata["tema"] == tema], embeddings)
    for tema in CLAVES_TEMA
}
rag_enrutado = RunnablePassthrough.assign(ruta=enrutador) | RunnablePassthrough.assign(
    docs=lambda x: almacenes[x["ruta"].fuente].similarity_search(x["pregunta"], k=2)
)

casos = {
    "¿Cómo planifica un agente una tarea larga?": "agentes",
    "¿Qué tamaño deben tener los fragmentos al indexar?": "rag",
    "¿Para qué sirve un juez en la evaluación?": "evaluacion",
}
for pregunta, esperado in casos.items():
    salida = rag_enrutado.invoke({"pregunta": pregunta})
    print(f"{pregunta:<52} -> {salida['ruta'].fuente:<10} {[d.metadata['titulo'] for d in salida['docs']]}")
    assert salida["ruta"].fuente == esperado

¿Cómo planifica un agente una tarea larga?           -> agentes    ['Agentes basados en LLM', 'Planificación y descomposición de tareas']
¿Qué tamaño deben tener los fragmentos al indexar?   -> rag        ['Indexación y troceado', 'Qué es RAG']
¿Para qué sirve un juez en la evaluación?            -> evaluacion ['Evaluación de aplicaciones LLM', 'Evaluación de aplicaciones LLM']


## 2. Enrutamiento semántico

En lugar de pedir la ruta al modelo, comparar el *embedding* de la pregunta con el de la descripción de cada experto y usar el prompt del más parecido. Mostrar la similitud con cada experto y comprobar que coincide con el enrutamiento lógico en las preguntas del ejercicio anterior.

In [3]:
EXPERTOS = {
    "agentes": "Experto en agentes autónomos: planificación de tareas, razonamiento, memoria, reflexión y uso de herramientas.",
    "rag": "Experto en RAG: indexación de documentos en fragmentos, embeddings, almacenes vectoriales, recuperación y reordenación.",
    "evaluacion": "Experto en evaluación: conjuntos de ejemplos, evaluadores, juez, métricas, experimentos y trazas con LangSmith.",
}
nombres = list(EXPERTOS)
vectores_expertos = np.array(embeddings.embed_documents(list(EXPERTOS.values())))
prompts_expertos = {
    nombre: ChatPromptTemplate.from_template(descripcion + " Responde con el contexto.\n\nContexto:\n{contexto}\n\nPregunta: {pregunta}")
    for nombre, descripcion in EXPERTOS.items()
}


def elegir_experto(entrada):
    similitudes = vectores_expertos @ np.array(embeddings.embed_query(entrada["pregunta"]))
    print("   ", {n: round(float(s), 2) for n, s in zip(nombres, similitudes)})
    return nombres[int(similitudes.argmax())]


semantico = RunnablePassthrough.assign(experto=elegir_experto) | RunnablePassthrough.assign(
    respuesta=lambda x: (prompts_expertos[x["experto"]] | ModeloLocal() | StrOutputParser()).invoke(
        {"pregunta": x["pregunta"], "contexto": formatear(recuperador.invoke(x["pregunta"]))}
    )
)
for pregunta, esperado in casos.items():
    salida = semantico.invoke({"pregunta": pregunta})
    print(f"{pregunta} -> {salida['experto']}: {salida['respuesta']}")
    assert salida["experto"] == esperado

    {'agentes': 0.47, 'rag': 0.0, 'evaluacion': 0.0}
¿Cómo planifica un agente una tarea larga? -> agentes: El agente combina planificación, memoria y uso de herramientas para resolver tareas largas.
    {'agentes': 0.0, 'rag': 0.28, 'evaluacion': 0.0}
¿Qué tamaño deben tener los fragmentos al indexar? -> rag: Para indexar, los documentos se dividen en fragmentos de unos cientos de caracteres con cierto solapamiento.
    {'agentes': 0.0, 'rag': 0.0, 'evaluacion': 0.46}
¿Para qué sirve un juez en la evaluación? -> evaluacion: Los evaluadores pueden ser reglas sencillas, métricas como la exactitud o un modelo que actúa como juez.


## 3. Construcción de consultas

Convertir una petición en lenguaje natural en una búsqueda estructurada `BusquedaArticulos` (texto de la consulta, tema, años mínimo y máximo, autor y visitas mínimas) y aplicar los campos como filtro de metadatos en un almacén con los artículos completos (`similarity_search(..., filter=función)`). El modelo local extrae los filtros con expresiones regulares y devuelve JSON. Probarlo con tres peticiones.

In [4]:
AUTORES = sorted({d.metadata["autor"] for d in documentos})


class BusquedaArticulos(BaseModel):
    consulta: str = Field(description="Texto que se busca por similitud")
    tema: Literal["agentes", "rag", "evaluacion"] | None = None
    anio_min: int | None = None
    anio_max: int | None = None
    autor: str | None = None
    min_vistas: int | None = None

    def admite(self, doc):
        m = doc.metadata
        return (
            (self.tema is None or m["tema"] == self.tema)
            and (self.anio_min is None or m["anio"] >= self.anio_min)
            and (self.anio_max is None or m["anio"] <= self.anio_max)
            and (self.autor is None or m["autor"] == self.autor)
            and (self.min_vistas is None or m["vistas"] > self.min_vistas)
        )


def estructurar(m):
    peticion = m.group(1)
    campos = {}
    if r := re.search(r"\b(?:de|sobre) (RAG|agentes|evaluación)\b", peticion):
        campos["tema"] = sin_tildes(r[1].lower())
    patrones = [
        (r"desde (\d{4})", lambda r: {"anio_min": int(r[1])}),
        (r"antes de (\d{4})", lambda r: {"anio_max": int(r[1]) - 1}),
        (r"en (\d{4})", lambda r: {"anio_min": int(r[1]), "anio_max": int(r[1])}),
        (r"más de (\d+) visitas", lambda r: {"min_vistas": int(r[1])}),
        (rf"({'|'.join(AUTORES)})", lambda r: {"autor": r[1]}),
    ]
    for patron, extraer in patrones:
        if r := re.search(patron, peticion):
            campos |= extraer(r)
            peticion = peticion.replace(r[0], " ")
    campos["consulta"] = re.sub(r"\s+", " ", re.sub(r"(?i)artículos|publicados|escritos por|con|qué escribió|[¿?]", " ", peticion)).strip()
    return json.dumps(campos, ensure_ascii=False)


parser_busqueda = PydanticOutputParser(pydantic_object=BusquedaArticulos)
constructor = (
    ChatPromptTemplate.from_template("Convierte la petición en una búsqueda estructurada.\n{formato}\n\nPetición: {peticion}").partial(
        formato=parser_busqueda.get_format_instructions()
    )
    | ModeloLocal(reglas=[(r"Petición:\s*(.*)", estructurar)])
    | parser_busqueda
)
almacen_articulos = InMemoryVectorStore.from_documents(documentos, embeddings)

for peticion in [
    "Artículos sobre la memoria de los agentes publicados antes de 2025",
    "¿Qué escribió Carmen Díaz sobre evaluación?",
    "Artículos de RAG desde 2025 con más de 500 visitas",
]:
    busqueda = constructor.invoke({"peticion": peticion})
    encontrados = almacen_articulos.similarity_search(busqueda.consulta, k=3, filter=busqueda.admite)
    print(f"{peticion}\n  {busqueda.model_dump(exclude_none=True)}")
    for d in encontrados:
        print(f"  - {d.metadata['titulo']} ({d.metadata['autor']}, {d.metadata['anio']}, {d.metadata['vistas']} visitas)")
        assert busqueda.admite(d)

Artículos sobre la memoria de los agentes publicados antes de 2025
  {'consulta': 'sobre la memoria de los agentes', 'anio_max': 2024}
  - Agentes basados en LLM (Ana Pérez, 2023, 1200 visitas)
  - Reflexión y autocrítica (Luis Santana, 2024, 610 visitas)
  - Memoria de los agentes (Luis Santana, 2024, 780 visitas)
¿Qué escribió Carmen Díaz sobre evaluación?
  {'consulta': 'sobre evaluación', 'tema': 'evaluacion', 'autor': 'Carmen Díaz'}
  - Evaluación de aplicaciones LLM (Carmen Díaz, 2025, 990 visitas)
Artículos de RAG desde 2025 con más de 500 visitas
  {'consulta': 'de RAG', 'tema': 'rag', 'anio_min': 2025, 'min_vistas': 500}
  - Transformación de consultas (Ana Pérez, 2025, 520 visitas)
  - Recuperación por similitud (Pedro Rodríguez, 2025, 640 visitas)


## 4. Indexación multirrepresentación

Indexar una representación pequeña de cada documento (un resumen) pero devolver el documento completo. Pedir al modelo un resumen de una frase de cada artículo (la regla elige la frase más central: la de mayor similitud media con las demás), guardar los resúmenes precedidos del título en un almacén vectorial con el `doc_id` en los metadatos y los artículos completos en un `InMemoryStore`. Comparar lo que devuelve esta búsqueda con la búsqueda por fragmentos.

In [5]:
from langchain_core.stores import InMemoryStore


def frase_central(texto, n=1):
    frases = frases_de(texto)
    if len(frases) <= n:
        return " ".join(frases)
    v = np.array(embeddings.embed_documents(frases))
    centralidad = (v @ v.T).sum(axis=1)
    elegidas = sorted(np.argsort(-centralidad)[:n])
    return " ".join(frases[i] for i in elegidas)


resumidor = ModeloLocal(reglas=[(r"Resume en (una|dos) frases?:\s*(.*)", lambda m: frase_central(m.group(2), 1 if m.group(1) == "una" else 2))])
resumir = ChatPromptTemplate.from_template("Resume en una frase: {texto}") | resumidor | StrOutputParser()

resumenes = resumir.batch([{"texto": d.page_content} for d in documentos])
almacen_resumenes = InMemoryVectorStore.from_documents(
    [Document(page_content=f"{d.metadata['titulo']}. {r}", metadata={"doc_id": d.metadata["id"]}) for r, d in zip(resumenes, documentos)],
    embeddings,
)
almacen_completos = InMemoryStore()
almacen_completos.mset([(d.metadata["id"], d) for d in documentos])


def multi_representacion(pregunta, k=1):
    ids = [r.metadata["doc_id"] for r in almacen_resumenes.similarity_search(pregunta, k=k)]
    return almacen_completos.mget(ids)


for d, r in list(zip(documentos, resumenes))[:3]:
    print(f"{d.metadata['titulo']}: {r}")
pregunta = "¿Qué propuestas enseñaron a los modelos a usar herramientas?"
print("\nResúmenes encontrados:", [r.page_content for r in almacen_resumenes.similarity_search(pregunta, k=2)])
completo = multi_representacion(pregunta)[0]
fragmento = recuperador.invoke(pregunta)[0]
print(f"Fragmento ({len(fragmento.page_content)} caracteres): {fragmento.page_content}")
print(f"Documento completo ({len(completo.page_content)} caracteres): {completo.page_content}")
assert completo.metadata["titulo"] == "Uso de herramientas" and fragmento.page_content in completo.page_content

Agentes basados en LLM: Un agente autónomo usa un modelo de lenguaje como cerebro.
Planificación y descomposición de tareas: La cadena de pensamiento pide al modelo que razone paso a paso antes de responder.
Reflexión y autocrítica: Reflexion guarda en memoria un resumen de los fallos anteriores para no repetirlos en el siguiente intento.

Resúmenes encontrados: ['Uso de herramientas. El modelo genera una llamada a herramienta con argumentos estructurados y el sistema la ejecuta.', 'Trazas y observabilidad. Una traza registra cada paso de una aplicación: las llamadas al modelo, las herramientas y los recuperadores.']
Fragmento (192 caracteres): El modelo genera una llamada a herramienta con argumentos estructurados y el sistema la ejecuta. Toolformer y MRKL son dos propuestas tempranas que enseñaron a los modelos a usar herramientas.
Documento completo (303 caracteres): Las herramientas amplían lo que un modelo puede hacer: calculadoras, buscadores, intérpretes de código o APIs. El mod

## 5. RAPTOR

RAPTOR construye un árbol de resúmenes: agrupa los textos con *clustering* sobre sus *embeddings*, resume cada grupo y repite con los resúmenes hasta que queda un solo grupo. Después indexa todos los niveles juntos (*collapsed tree*), de modo que las preguntas generales encuentran resúmenes y las concretas, textos originales. Construir el árbol a partir de los artículos con `KMeans` (un grupo por cada 3 textos) y el resumidor de dos frases, y mostrar de qué nivel salen los resultados de una pregunta concreta y de una general.

In [6]:
from sklearn.cluster import KMeans

resumir_grupo = ChatPromptTemplate.from_template("Resume en dos frases: {texto}") | resumidor | StrOutputParser()
nodos = [Document(page_content=d.page_content, metadata={"nivel": 0}) for d in documentos]
nivel_actual = nodos
for nivel in range(1, 4):
    if len(nivel_actual) <= 2:
        break
    v = np.array(embeddings.embed_documents([d.page_content for d in nivel_actual]))
    k = max(1, len(nivel_actual) // 3)
    grupos = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(v)
    textos = [" ".join(d.page_content for d, g in zip(nivel_actual, grupos) if g == i) for i in range(k)]
    nivel_actual = [Document(page_content=r, metadata={"nivel": nivel}) for r in resumir_grupo.batch([{"texto": t} for t in textos])]
    nodos += nivel_actual
    print(f"Nivel {nivel}: {len(nivel_actual)} resúmenes, tamaños de grupo {sorted(Counter(grupos).values(), reverse=True)}")

print("Raíz:", nivel_actual[0].page_content if len(nivel_actual) == 1 else [d.page_content for d in nivel_actual])
almacen_raptor = InMemoryVectorStore.from_documents(nodos, embeddings)
for pregunta in ["¿Qué algoritmos aceleran la búsqueda en millones de vectores?", "¿Qué técnicas mejoran la recuperación?"]:
    print(f"\n{pregunta}")
    for d in almacen_raptor.similarity_search(pregunta, k=3):
        print(f"  nivel {d.metadata['nivel']}: {d.page_content[:100]}...")

Nivel 1: 4 resúmenes, tamaños de grupo [3, 3, 3, 3]


Nivel 2: 1 resúmenes, tamaños de grupo [4]
Raíz: Una traza registra cada paso de una aplicación: las llamadas al modelo, las herramientas y los recuperadores. El modelo genera una llamada a herramienta con argumentos estructurados y el sistema la ejecuta.

¿Qué algoritmos aceleran la búsqueda en millones de vectores?
  nivel 0: La memoria a corto plazo es el contexto de la conversación que cabe en la ventana del modelo. La mem...
  nivel 1: Los vectores se guardan en un almacén vectorial junto con el texto y sus metadatos. La recuperación ...
  nivel 0: La recuperación compara el vector de la pregunta con los vectores de los fragmentos. La similitud de...

¿Qué técnicas mejoran la recuperación?
  nivel 0: Un reordenador vuelve a puntuar los fragmentos recuperados con un modelo más preciso y se queda con ...
  nivel 1: La generación aumentada por recuperación, o RAG, conecta un modelo de lenguaje con documentos extern...
  nivel 0: La generación aumentada por recuperación, o RAG, conec

## 6. ColBERT

ColBERT no resume cada texto en un solo vector: guarda un vector por palabra y puntúa con *late interaction* (MaxSim), sumando para cada palabra de la pregunta su máxima similitud con las palabras del documento. Implementarlo con vectores de n-gramas de caracteres por palabra, de modo que tolere erratas y variantes que la raíz de 5 letras no reconoce (*alusinaciones*, *entrenar* frente a *reentrenar*). Comparar la posición del fragmento correcto con la búsqueda de un vector por texto.

In [7]:
vectorizador_palabras = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 4), n_features=2**12, alternate_sign=False, norm="l2")


def matriz_palabras(texto):
    palabras = [p for p in re.findall(r"\w+", sin_tildes(texto.lower())) if p not in VACIAS and len(p) > 2]
    return vectorizador_palabras.transform(palabras).toarray()


matrices = [matriz_palabras(t.page_content) for t in trozos]


def maxsim(pregunta):
    q = matriz_palabras(pregunta)
    return np.array([(q @ m.T).max(axis=1).sum() for m in matrices])


def posicion(puntuaciones, condicion):
    orden = np.argsort(-puntuaciones, kind="stable")
    return next(i for i, j in enumerate(orden, 1) if condicion(trozos[j].page_content))


vectores_trozos = np.array(embeddings.embed_documents([t.page_content for t in trozos]))
for pregunta, palabra in [("¿Qué evita las alusinaciones?", "alucinaciones"), ("¿Hace falta entrenar otra vez el modelo?", "reentrenar")]:
    un_vector = vectores_trozos @ np.array(embeddings.embed_query(pregunta))
    p1 = posicion(un_vector, lambda t: palabra in t)
    p2 = posicion(maxsim(pregunta), lambda t: palabra in t)
    print(f"{pregunta:<42} un vector: posición {p1:>2} | ColBERT: posición {p2}")
    assert p2 <= p1

¿Qué evita las alusinaciones?              un vector: posición 13 | ColBERT: posición 1
¿Hace falta entrenar otra vez el modelo?   un vector: posición  5 | ColBERT: posición 1


## 7. Reordenación

Recuperar 8 candidatos por similitud y reordenarlos con BM25, una puntuación léxica que tiene en cuenta la frecuencia de cada término, lo raro que es en el corpus (IDF) y la longitud del fragmento. Implementar el reordenador como un `BaseDocumentCompressor` que añade `relevance_score` a los metadatos y se queda con los 3 mejores, y mostrar el orden antes y después.

In [8]:
from langchain_core.documents import BaseDocumentCompressor

tokens_corpus = [tokens(t.page_content) for t in trozos]
frecuencia_doc = Counter(tok for ts in tokens_corpus for tok in set(ts))
N = len(tokens_corpus)


class ReordenadorBM25(BaseDocumentCompressor):
    top_n: int = 3
    k1: float = 1.5
    b: float = 0.75
    idf: dict[str, float] = {tok: np.log(1 + (N - n + 0.5) / (n + 0.5)) for tok, n in frecuencia_doc.items()}
    longitud_media: float = sum(map(len, tokens_corpus)) / N

    def puntuar(self, consulta, texto):
        ts = tokens(texto)
        tf = Counter(ts)
        norma = self.k1 * (1 - self.b + self.b * len(ts) / self.longitud_media)
        return sum(self.idf.get(q, 0) * tf[q] * (self.k1 + 1) / (tf[q] + norma) for q in set(tokens(consulta)))

    def compress_documents(self, documents, query, callbacks=None):
        puntuados = sorted(((self.puntuar(query, d.page_content), d) for d in documents), key=lambda x: -x[0])
        return [Document(page_content=d.page_content, metadata={**d.metadata, "relevance_score": round(float(p), 3)}) for p, d in puntuados[: self.top_n]]


pregunta = "¿Qué hace el modelo con los argumentos de una llamada a herramienta?"
candidatos = almacen.similarity_search(pregunta, k=8)
print("Antes:  ", [c.metadata["titulo"] for c in candidatos[:3]])
reordenados = ReordenadorBM25().compress_documents(candidatos, pregunta)
print("Después:", [(d.metadata["titulo"], d.metadata["relevance_score"]) for d in reordenados])
print(reordenados[0].page_content)
assert "argumentos" in reordenados[0].page_content

Antes:   ['Uso de herramientas', 'Uso de herramientas', 'Trazas y observabilidad']
Después: [('Uso de herramientas', 7.296), ('Trazas y observabilidad', 4.339), ('Uso de herramientas', 2.521)]
El modelo genera una llamada a herramienta con argumentos estructurados y el sistema la ejecuta. Toolformer y MRKL son dos propuestas tempranas que enseñaron a los modelos a usar herramientas.


## 8. RAG correctivo (CRAG) con LangGraph

Construir con `StateGraph` un flujo que recupera, califica la relevancia de cada documento y, si ninguno es relevante, reescribe la consulta y busca en la "web" (una lista de textos externos) antes de generar. El calificador es el modelo local respondiendo `{"relevante": "si" | "no"}` según la proporción de raíces de la pregunta que aparecen en el documento. Mostrar el diagrama del grafo en Mermaid y el camino que sigue cada una de dos preguntas: una del corpus y otra que no está en él.

In [9]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph

WEB = [
    "LangGraph es una librería de LangChain para construir agentes y flujos con estado como grafos de nodos y aristas.",
    "LangSmith es la plataforma de LangChain para trazar, evaluar y monitorizar aplicaciones con modelos de lenguaje.",
    "El Teide, en Tenerife, es el pico más alto de España con 3715 metros.",
]
almacen_web = InMemoryVectorStore.from_texts(WEB, embeddings, metadatas=[{"titulo": "web"}] * len(WEB))


def juzgar(m):
    documento, pregunta = set(tokens(m.group(1))), set(tokens(m.group(2)))
    return json.dumps({"relevante": "si" if pregunta and len(documento & pregunta) / len(pregunta) >= 0.5 else "no"})


SINONIMOS = {"reranker": "reordenador", "alusinaciones": "alucinaciones", "langgraph": "LangGraph librería grafos"}


def reescribir(m):
    claves = [SINONIMOS.get(p.lower(), p) for p in re.findall(r"\w+", m.group(1)) if sin_tildes(p.lower()) not in VACIAS]
    return " ".join(claves)


modelo = ModeloLocal(reglas=[
    (r"¿Es relevante el documento.*?Documento:\s*(.*?)\s*Pregunta:\s*(.*)", juzgar),
    (r"Reescribe la pregunta para un buscador:\s*(.*)", reescribir),
])
calificador = ChatPromptTemplate.from_template("¿Es relevante el documento para la pregunta? Responde en JSON.\nDocumento: {documento}\nPregunta: {pregunta}") | modelo | StrOutputParser() | json.loads
reescritor = ChatPromptTemplate.from_template("Reescribe la pregunta para un buscador: {pregunta}") | modelo | StrOutputParser()
generador = prompt_rag | modelo | StrOutputParser()


class EstadoRAG(TypedDict, total=False):
    pregunta: str
    consulta: str
    documentos: list[Document]
    buscar_en_web: bool
    respuesta: str
    pasos: Annotated[list[str], operator.add]


def recuperar(estado):
    return {"documentos": recuperador.invoke(estado.get("consulta") or estado["pregunta"]), "pasos": ["recuperar"]}


def calificar(estado):
    relevantes = [d for d in estado["documentos"] if calificador.invoke({"documento": d.page_content, "pregunta": estado["pregunta"]})["relevante"] == "si"]
    return {"documentos": relevantes, "buscar_en_web": not relevantes, "pasos": [f"calificar ({len(relevantes)} relevantes)"]}


def transformar(estado):
    return {"consulta": reescritor.invoke({"pregunta": estado["pregunta"]}), "pasos": ["transformar"]}


def buscar_web(estado):
    return {"documentos": almacen_web.similarity_search(estado["consulta"], k=1), "pasos": ["buscar_web"]}


def generar(estado):
    pregunta = estado.get("consulta") or estado["pregunta"]
    return {"respuesta": generador.invoke({"contexto": formatear(estado["documentos"]), "pregunta": pregunta}), "pasos": ["generar"]}


flujo = StateGraph(EstadoRAG)
for nodo in (recuperar, calificar, transformar, buscar_web, generar):
    flujo.add_node(nodo.__name__, nodo)
flujo.add_edge(START, "recuperar")
flujo.add_edge("recuperar", "calificar")
flujo.add_conditional_edges("calificar", lambda e: "transformar" if e["buscar_en_web"] else "generar", ["transformar", "generar"])
flujo.add_edge("transformar", "buscar_web")
flujo.add_edge("buscar_web", "generar")
flujo.add_edge("generar", END)
crag = flujo.compile()
print(crag.get_graph().draw_mermaid())

for pregunta in ["¿Qué evalúa el RAG correctivo?", "¿Qué es LangGraph?"]:
    final = crag.invoke({"pregunta": pregunta, "pasos": []})
    print(f"{pregunta}\n  camino: {' -> '.join(final['pasos'])}\n  respuesta: {final['respuesta']}")
assert "buscar_web" in final["pasos"] and "LangGraph" in final["respuesta"]

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	recuperar(recuperar)
	calificar(calificar)
	transformar(transformar)
	buscar_web(buscar_web)
	generar(generar)
	__end__([<p>__end__</p>]):::last
	__start__ --> recuperar;
	buscar_web --> generar;
	calificar -.-> generar;
	calificar -.-> transformar;
	recuperar --> calificar;
	transformar --> buscar_web;
	generar --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

¿Qué evalúa el RAG correctivo?
  camino: recuperar -> calificar (1 relevantes) -> generar
  respuesta: El RAG correctivo, CRAG, evalúa si los documentos recuperados son relevantes.
¿Qué es LangGraph?
  camino: recuperar -> calificar (0 relevantes) -> transformar -> buscar_web -> generar
  respuesta: LangGraph es una librería de LangChain para construir agentes y flujos con estado como grafos de nodos y aristas.


## 9. Self-RAG con LangGraph

Añadir al flujo una comprobación de la respuesta: que esté **fundamentada** (todas sus raíces aparecen en los documentos) y que sea **útil** (comparte raíces con la pregunta y no es "No lo sé"). Si no es útil, reescribir la pregunta y volver a recuperar, con un máximo de 2 reintentos. Probarlo con una pregunta que usa una palabra que no está en el corpus (*reranker*) y mostrar el camino.

In [10]:
def comprobar(estado):
    respuesta, docs = estado["respuesta"], formatear(estado["documentos"])
    fundamentada = set(tokens(respuesta)) <= set(tokens(docs))
    util = respuesta != "No lo sé." and bool(set(tokens(respuesta)) & set(tokens(estado.get("consulta") or estado["pregunta"])))
    return {"pasos": [f"comprobar (fundamentada={fundamentada}, útil={util})"], "buscar_en_web": not (fundamentada and util)}


def calificar_consulta(estado):
    pregunta = estado.get("consulta") or estado["pregunta"]
    relevantes = [d for d in estado["documentos"] if calificador.invoke({"documento": d.page_content, "pregunta": pregunta})["relevante"] == "si"]
    return {"documentos": relevantes, "pasos": [f"calificar ({len(relevantes)} relevantes)"]}


def siguiente(estado):
    if not estado["buscar_en_web"]:
        return END
    return "transformar" if estado["pasos"].count("transformar") < 2 else END


self_rag = StateGraph(EstadoRAG)
for nombre, nodo in [("recuperar", recuperar), ("calificar", calificar_consulta), ("generar", generar), ("comprobar", comprobar), ("transformar", transformar)]:
    self_rag.add_node(nombre, nodo)
self_rag.add_edge(START, "recuperar")
self_rag.add_edge("recuperar", "calificar")
self_rag.add_edge("calificar", "generar")
self_rag.add_edge("generar", "comprobar")
self_rag.add_conditional_edges("comprobar", siguiente, ["transformar", END])
self_rag.add_edge("transformar", "recuperar")
self_rag = self_rag.compile()

for pregunta in ["¿Qué mide la similitud del coseno?", "¿Qué hace un reranker?"]:
    final = self_rag.invoke({"pregunta": pregunta, "pasos": []})
    print(f"{pregunta}\n  camino: {' -> '.join(final['pasos'])}\n  respuesta: {final['respuesta']}")
assert final["pasos"].count("transformar") == 1 and "reordenador" in final["respuesta"]

¿Qué mide la similitud del coseno?
  camino: recuperar -> calificar (1 relevantes) -> generar -> comprobar (fundamentada=True, útil=True)
  respuesta: La similitud del coseno mide el ángulo entre dos vectores y es la métrica más habitual.
¿Qué hace un reranker?
  camino: recuperar -> calificar (0 relevantes) -> generar -> comprobar (fundamentada=True, útil=False) -> transformar -> recuperar -> calificar (1 relevantes) -> generar -> comprobar (fundamentada=True, útil=True)
  respuesta: Un reordenador vuelve a puntuar los fragmentos recuperados con un modelo más preciso y se queda con los mejores.


## 10. Contexto largo frente a RAG

Con ventanas de contexto grandes se puede meter el corpus entero en el prompt en lugar de recuperar. Para varias preguntas, comparar el tamaño del contexto (en caracteres y en raíces) y la respuesta usando todo el corpus o solo los 4 fragmentos recuperados.

In [11]:
todo = formatear(documentos)
for pregunta in ["¿Qué reduce RAG?", "¿Qué guarda Reflexion en memoria?", "¿Qué registra una traza?"]:
    recuperado = formatear(recuperador.invoke(pregunta))
    largo = (prompt_rag | ModeloLocal() | StrOutputParser()).invoke({"contexto": todo, "pregunta": pregunta})
    corto = (prompt_rag | ModeloLocal() | StrOutputParser()).invoke({"contexto": recuperado, "pregunta": pregunta})
    print(f"{pregunta}\n  todo el corpus: {len(todo)} caracteres, {len(tokens(todo))} raíces -> {largo}")
    print(f"  RAG:            {len(recuperado)} caracteres, {len(tokens(recuperado))} raíces -> {corto}")

¿Qué reduce RAG?
  todo el corpus: 3429 caracteres, 298 raíces -> RAG reduce las alucinaciones y permite usar información privada o reciente sin reentrenar el modelo.
  RAG:            564 caracteres, 51 raíces -> RAG reduce las alucinaciones y permite usar información privada o reciente sin reentrenar el modelo.
¿Qué guarda Reflexion en memoria?
  todo el corpus: 3429 caracteres, 298 raíces -> Reflexion guarda en memoria un resumen de los fallos anteriores para no repetirlos en el siguiente intento.
  RAG:            549 caracteres, 48 raíces -> Reflexion guarda en memoria un resumen de los fallos anteriores para no repetirlos en el siguiente intento.
¿Qué registra una traza?
  todo el corpus: 3429 caracteres, 298 raíces -> Una traza registra cada paso de una aplicación: las llamadas al modelo, las herramientas y los recuperadores.
  RAG:            548 caracteres, 46 raíces -> Una traza registra cada paso de una aplicación: las llamadas al modelo, las herramientas y los recuperadores